# Continual learning

> The gain predictor learning on over a stream of layouts it meets one after another, as in continual learning for
> channel prediction (Mohsin et al., ICML 2025): fine-tuning, experience replay (reservoir and loss-aware), EWC, SI
> and learning without forgetting, against the frozen predictor, joint training and the trees fit again.

In [ ]:
#| default_exp comm.continual

In [ ]:
#| hide
from nbdev.showdoc import *

[Mohsin et al.](https://cioffi-group.stanford.edu/mlc/recents/continual_learning/paper.pdf) train a channel predictor
(an LSTM, a GRU, a Transformer) on one propagation scenario after another, and compare ways to learn each new one
without forgetting the earlier ones: replaying stored samples, penalizing the weights that mattered before, or
distilling the earlier model. Here the same, for C3-WM's gain predictor (`comm.sequence`):

* **the tasks**: layouts, met one after another (`stream_order`: e.g. the deployment layouts, in-distribution, then
  slightly OOD, then OOD, `comm.deployment`); the predictor is pre-trained on the training layouts (the first task);
* **the feedback**: in each layout, the gains measured over its first `update_episodes` episodes (what the receivers
  report), with the features they were predicted from (`layout_stream`); the layout's other episodes are sent with
  the predictor after it learned from them;
* **the methods** (`ContinualLearner`): as the paper's, with its settings (a buffer of 5000 samples, `α = β = 0.4`,
  `ξ = 0.1`, `λ = 0.5`), the loss the predictor's own (the pinball loss of its quantile, or the squared error);
  the trees, which no gradient updates, are fit again on everything seen (`TreesRefit`);
* **the judging** (`run_stream`): the reliability and power of each layout's transmissions, each one sent with the
  `κ` and `τ` chosen for the predictor as it then is on the calibration layouts (`adapted`), or with the start's
  (`fixed`); and the final predictor on every layout of the stream (`final`: what it forgot of the first ones) and on
  the test layouts (`test`: of the training layouts' kind).

In [ ]:
#| export
from __future__ import annotations

import copy
import math

import numpy as np

from c3jepa_wm.comm.deployment import judge
from c3jepa_wm.comm.evaluation import prediction_errors, transmissions
from c3jepa_wm.comm.power_control import TAUS_DB, calibrate, fit_predictor, predict_on
from c3jepa_wm.comm.sequence import GEOMETRY, SequencePredictor, fit_model, gain_loss, predictor_name

METHODS = ('frozen', 'finetune', 'replay', 'lars', 'ewc', 'si', 'lwf', 'joint')
SAMPLES = ('X', 'S', 'gain')                                        # what a learner learns from


def subset(data: dict, keep: np.ndarray) -> dict:
    "The samples of `data` (`build_dataset`'s) where `keep`."
    return {k: v[keep] for k, v in data.items()}

## The replay buffer

In [ ]:
#| export
class ReplayBuffer:
    """Past samples (features, past steps, gain), at most `capacity` (None: all): kept by reservoir sampling, so every
    sample seen is equally likely to be kept; with `lars`, the one replaced is drawn with probability ∝ 1 / (loss + ε)
    (loss-aware reservoir sampling: the well-predicted samples leave first, the hard ones stay)."""

    def __init__(self, capacity: int | None = 5000, lars: bool = False, seed: int = 0, eps: float = 1e-8):
        self.capacity, self.lars, self.eps = capacity, lars, eps
        self.rng = np.random.default_rng(seed)
        self.seen, self.n, self.arrays = 0, 0, None

    def __len__(self):
        return self.n

    def _grow(self, rows: dict, need: int):
        size = need if self.capacity is not None else max(need, 2 * (len(self.arrays['gain']) if self.arrays else 0), 1024)
        new = {k: np.zeros((size, *v.shape[1:]), v.dtype) for k, v in rows.items()}
        if self.arrays is not None:
            for k in new:
                new[k][:self.n] = self.arrays[k][:self.n]
        self.arrays = new

    def add(self, X: np.ndarray, gain: np.ndarray, loss: np.ndarray, S: np.ndarray | None = None):
        "Offer each sample (its `loss` under the current model) to the buffer."
        rows = dict(X=np.asarray(X, np.float32), gain=np.asarray(gain, np.float64), loss=np.asarray(loss, np.float64))
        if S is not None:
            rows['S'] = np.asarray(S, np.float32)
        if self.arrays is None or (self.capacity is None and self.n + len(gain) > len(self.arrays['gain'])):
            self._grow(rows, self.capacity if self.capacity is not None else self.n + len(gain))
        for i in range(len(gain)):
            self.seen += 1
            if self.capacity is None or self.n < self.capacity:
                slot = self.n
                self.n += 1
            else:
                j = int(self.rng.integers(self.seen))
                if j >= self.capacity:
                    continue
                if self.lars:
                    w = 1 / (self.arrays['loss'][:self.n] + self.eps)
                    j = int(self.rng.choice(self.n, p=w / w.sum()))
                slot = j
            for k, v in rows.items():
                self.arrays[k][slot] = v[i]

    def sample(self, n: int) -> dict:
        "`n` samples drawn uniformly (with replacement)."
        idx = self.rng.integers(self.n, size=n)
        return {k: v[idx] for k, v in self.arrays.items() if k != 'loss'}

## The learners

In [ ]:
#| export
class ContinualLearner:
    """A sequence predictor (`comm.sequence`) learning on, layout after layout, by one of `METHODS`:

    * `frozen`: never updated (the offline predictor);
    * `finetune`: each layout's samples alone (forgets);
    * `replay`, `lars`: each batch mixed with as many samples of a replay buffer (`ReplayBuffer`: reservoir, or
      loss-aware), the training samples in it from the start; the loss `w L_current + (1 - w) L_replay`;
    * `ewc`: a penalty `α Σ_j Σ_i F_j,i (θ_i - θ*_j,i)²`, the Fisher information of each past task (the training's
      first) at its weights;
    * `si`: a penalty `β Σ_i ω_i (θ_i - θ⁰_i)²`, each weight's importance accumulated along the training (`ω`, from
      the predictor's own training: `SequencePredictor.importance`) and every layout since;
    * `lwf`: the loss plus `λ` times the squared difference to the predictions of the model before this layout;
    * `joint`: replay of every sample seen (the training's and every layout's): the bound the others approach.

    It predicts as the predictor does (`predict`, `steps`), so `predict_on` and the calibration take it."""

    def __init__(
        self,
        model: SequencePredictor, # The pre-trained predictor (copied: it is not changed)
        method: str = 'replay', # One of `METHODS`
        train: dict | None = None, # Its training samples (`build_dataset`'s): replay's first contents, EWC's first task
        capacity: int = 5000, # The replay buffer's (replay, lars)
        epochs: int = 20, # Over each layout's samples
        batch_size: int = 64,
        lr: float = 5e-4,
        replay_weight: float = 0.5, # w: of the current layout's loss (replay, lars, joint)
        ewc: float = 0.4, # α
        si: float = 0.4, # β
        si_xi: float = 0.1, # ξ: SI's damping
        lwf: float = 0.5, # λ
        clip: float = 1.0, # The gradients' norm
        fisher_samples: int = 20000, # At most, for EWC's Fisher information
        device: str = 'auto',
        seed: int = 0,
    ):
        import torch
        if method not in METHODS:
            raise ValueError(f'method: one of {METHODS}, not {method!r}')
        if not isinstance(model, SequencePredictor):
            raise ValueError('a ContinualLearner learns a sequence predictor (comm.sequence); the trees: TreesRefit')
        if method in ('replay', 'lars', 'joint', 'ewc') and train is None:
            raise ValueError(f'{method} needs the training samples (train)')
        if method == 'si' and model.importance is None:
            raise ValueError("si needs the predictor's importance over its training (fit_sequence_predictor keeps it)")
        self.model, self.method, self._torch = copy.deepcopy(model), method, torch
        self.epochs, self.batch_size, self.lr, self.replay_weight = epochs, batch_size, lr, replay_weight
        self.alpha, self.beta, self.xi, self.lwf, self.clip = ewc, si, si_xi, lwf, clip
        self.fisher_samples = fisher_samples
        self.device = ('cuda' if torch.cuda.is_available() else 'cpu') if device == 'auto' else device
        self.rng = np.random.default_rng(seed)
        torch.manual_seed(seed)
        self.loss = gain_loss(model.quantile)
        self.tasks = 0
        if method in ('replay', 'lars', 'joint'):
            self.buffer = ReplayBuffer(None if method == 'joint' else capacity, method == 'lars', seed)
            self.buffer.add(train['X'], train['gain'], self.losses(train), train.get('S') if model.steps else None)
        if method == 'ewc':
            self.anchors = [(self._weights(), self._fisher(train))]
        if method == 'si':
            self.omega = {k: v.clone() for k, v in model.importance.items()}
            self.anchor = self._weights()

    @property
    def steps(self) -> bool:
        return self.model.steps

    def predict(self, X: np.ndarray, S: np.ndarray | None = None) -> np.ndarray:
        "The gain (dB), as `SequencePredictor.predict`."
        return self.model.predict(X, S) if self.steps else self.model.predict(X)

    def _weights(self) -> dict:
        return {k: p.detach().clone() for k, p in self.model.net.named_parameters()}

    def _tensors(self, data: dict) -> tuple:
        torch = self._torch
        Z, Zs = self.model.scaled(data['X'], data.get('S'))
        t = self.model.target(data['gain'])
        to = lambda a: torch.from_numpy(a).to(self.device) if a is not None else None
        return to(Z), to(Zs), to(t)

    def losses(self, data: dict) -> np.ndarray:
        "Each sample's loss under the current model (standardized dB): what the loss-aware buffer keeps the hard ones by."
        c, q = self.model.scale, self.model.quantile
        p = (self.predict(data['X'], data.get('S')) - c['mean'][GEOMETRY]) / c['std'][GEOMETRY]
        e = self.model.target(data['gain']) - p
        return (e ** 2 if q is None else np.maximum(q * e, (q - 1) * e)).astype(float)

    def _fisher(self, data: dict) -> dict:
        "The diagonal Fisher information of the loss at the current weights: the mean squared gradient over batches."
        torch = self._torch
        n = len(data['gain'])
        keep = self.rng.choice(n, min(n, self.fisher_samples), replace=False)
        Z, Zs, t = self._tensors(subset({k: v for k, v in data.items() if k in SAMPLES}, keep))
        net = copy.deepcopy(self.model.net).to(self.device)
        net.train()
        F = {k: torch.zeros_like(p) for k, p in net.named_parameters()}
        batches = 0
        for i in range(0, len(t), self.batch_size):
            net.zero_grad()
            self.loss(net(Z[i:i + self.batch_size], Zs[i:i + self.batch_size] if Zs is not None else None),
                      t[i:i + self.batch_size]).backward()
            for k, p in net.named_parameters():
                if p.grad is not None:
                    F[k] += p.grad.detach() ** 2
            batches += 1
        return {k: (v / max(batches, 1)).cpu() for k, v in F.items()}

    def learn(self, task: dict):
        "Learn from one layout's samples (`build_dataset`'s): its feedback, the gains measured."
        self.tasks += 1
        if self.method == 'frozen' or not len(task['gain']):
            return
        torch, dev = self._torch, self.device
        task = {k: v for k, v in task.items() if k in SAMPLES and (k != 'S' or self.steps)}
        net = copy.deepcopy(self.model.net).to(dev)
        teacher = copy.deepcopy(net).eval() if self.method == 'lwf' else None
        net.train()
        opt = torch.optim.Adam(net.parameters(), lr=self.lr)
        Z, Zs, t = self._tensors(task)
        anchors = [({k: v.to(dev) for k, v in a.items()}, {k: v.to(dev) for k, v in F.items()})
                   for a, F in getattr(self, 'anchors', [])]
        if self.method == 'si':
            omega = {k: v.to(dev) for k, v in self.omega.items()}
            anchor = {k: v.to(dev) for k, v in self.anchor.items()}
            path = {k: torch.zeros_like(p) for k, p in net.named_parameters()}
        replay = self.method in ('replay', 'lars', 'joint') and len(self.buffer) > 0
        n = len(t)
        for _ in range(self.epochs):
            order = torch.randperm(n, device=dev)
            for i in range(0, n, self.batch_size):
                b = order[i:i + self.batch_size]
                out = net(Z[b], Zs[b] if Zs is not None else None)
                loss = self.loss(out, t[b])
                if replay:
                    Rz, Rzs, Rt = self._tensors(self.buffer.sample(len(b)))
                    loss = self.replay_weight * loss + (1 - self.replay_weight) * self.loss(net(Rz, Rzs), Rt)
                if teacher is not None:
                    with torch.no_grad():
                        target = teacher(Z[b], Zs[b] if Zs is not None else None)
                    loss = loss + self.lwf * ((out - target) ** 2).mean()
                params = dict(net.named_parameters())
                if anchors:
                    loss = loss + self.alpha * sum((F[k] * (params[k] - a[k]) ** 2).sum() for a, F in anchors for k in F)
                if self.method == 'si':
                    loss = loss + self.beta * sum((omega[k] * (params[k] - anchor[k]) ** 2).sum() for k in omega)
                opt.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), self.clip)
                if self.method == 'si':
                    for k, p in params.items():
                        if p.grad is not None:
                            path[k] += p.grad.detach() ** 2 * self.lr
                opt.step()
        self.model.loaded(net.state_dict())
        if self.method in ('replay', 'lars', 'joint'):
            self.buffer.add(task['X'], task['gain'], self.losses(task), task.get('S'))
        if self.method == 'ewc':
            self.anchors.append((self._weights(), self._fisher(task)))
        if self.method == 'si':
            now = self._weights()
            self.omega = {k: (omega[k] + path[k] / ((now[k].to(dev) - anchor[k]) ** 2 + self.xi)).cpu() for k in omega}
            self.anchor = now

In [ ]:
#| export
class TreesRefit:
    """Gradient-boosted trees learning on the only way they can: fit again, every `every` layouts, on the training
    samples and every layout's so far (`every=None`: never, the frozen trees)."""
    steps = False

    def __init__(self, model, train: dict, quantile: float | None, every: int | None = 10, **trees):
        self.model, self.quantile, self.every, self.trees = model, quantile, every, trees
        self.seen = [{k: train[k] for k in ('X', 'gain')}]
        self.tasks = 0

    def predict(self, X: np.ndarray, S: np.ndarray | None = None) -> np.ndarray:
        return self.model.predict(X)

    def learn(self, task: dict):
        self.tasks += 1
        if not self.every:
            return
        self.seen.append({k: task[k] for k in ('X', 'gain')})
        if self.tasks % self.every == 0:
            X, gain = (np.concatenate([s[k] for s in self.seen]) for k in ('X', 'gain'))
            self.model = fit_predictor(X, gain, self.quantile, **self.trees)

## The stream

In [ ]:
#| export
def layout_stream(data: dict, order, update_episodes: int = 2) -> list[dict]:
    """Per layout of `order`: its samples of the first `update_episodes` episodes (`update`: the feedback a learner
    learns from) and of the others (`judged`)."""
    out = []
    for k in order:
        on = data['layout'] == k
        out.append(dict(layout=int(k), update=subset(data, on & (data['episode'] < update_episodes)),
                        judged=subset(data, on & (data['episode'] >= update_episodes))))
    return out


def stream_order(groups: dict, order: str = 'groups', seed: int = 0) -> list[int]:
    """The layouts in the order they come: `groups`, group after group as given (e.g. in-distribution, slightly OOD,
    OOD), each shuffled; `random`, all shuffled."""
    rng = np.random.default_rng(seed)
    if order == 'groups':
        return [int(k) for ids in groups.values() for k in rng.permutation(list(ids))]
    if order == 'random':
        return [int(k) for k in rng.permutation([k for ids in groups.values() for k in ids])]
    raise ValueError(f"order: groups or random, not {order!r}")


def _params(learner, calibration: dict, g_min: float, targets, taus_db, slack) -> dict:
    "Per target: κ and τ chosen for the learner as it is now, on the calibration layouts."
    estimate = predict_on(learner, calibration)
    return {t: calibrate(estimate, calibration['gain'], g_min, t + slack, taus_db=taus_db) for t in targets}


def _judged(estimates, gains, layouts, episodes, params, g_min, target, members) -> dict:
    "`judge`'s metrics per group of the transmissions, each part sent with its own κ and τ (`params`, one per part)."
    power, sent = map(np.concatenate, zip(*[transmissions(e, g_min, p['margin_db'], p['tau_db']) for e, p in zip(estimates, params)]))
    gain, layout, episode = (np.concatenate(a) for a in (gains, layouts, episodes))
    return {g: judge(power[m], sent[m], gain[m], layout[m], g_min, target, episode=episode[m])
            for g, ids in members.items() for m in [np.isin(layout, ids)] if m.any()}


def run_stream(
    learners: dict, # name -> a learner (`ContinualLearner`, `TreesRefit`): learns on in place
    tasks: list, # `layout_stream`'s
    calibration: dict, # The calibration layouts' samples: κ and τ chosen there after each update
    test: dict, # The test layouts' samples: what the final predictor still knows of them (forgetting)
    g_min: float, # The weakest servable gain
    targets=(0.9, 0.95, 0.99), # Reliability targets
    taus_db=TAUS_DB, slack: float = 0.0, # As the calibration's
    groups: dict | None = None, # Group -> its layouts (None: one group of them all)
    recalibrate_every: int = 1, # Layouts between choosing κ and τ again
) -> dict:
    """Per learner, layout after layout: it learns from the layout's feedback, then the layout's other episodes are
    sent with its prediction and the κ and τ chosen for it on the calibration layouts. `tasks`: per layout its
    prediction errors before and after learning from it, and the metrics at each target after; `targets`: per target
    and group, the metrics (`judge`'s) of the stream pooled, `adapted` (κ and τ chosen again) and `fixed` (the start's
    κ and τ kept), and of the final predictor, on every layout of the stream (`final`) and on the test layouts (`test`)."""
    members = {'all': [t['layout'] for t in tasks], **(groups or {})}
    group_of = {k: g for g, ids in (groups or {}).items() for k in ids}
    out = {}
    for name, learner in learners.items():
        start = _params(learner, calibration, g_min, targets, taus_db, slack)
        params, rows, est = start, [], []
        for i, task in enumerate(tasks):
            j = task['judged']
            before = predict_on(learner, j)
            learner.learn(task['update'])
            if (i + 1) % recalibrate_every == 0:
                params = _params(learner, calibration, g_min, targets, taus_db, slack)
            after = predict_on(learner, j)
            est.append((after, dict(params)))
            row = dict(task=i, layout=task['layout'], group=group_of.get(task['layout'], 'all'), transmissions=len(j['gain']),
                       before=prediction_errors(before, j['gain']), after=prediction_errors(after, j['gain']))
            for t in targets:
                power, sent = transmissions(after, g_min, params[t]['margin_db'], params[t]['tau_db'])
                row[t] = {k: v for k, v in judge(power, sent, j['gain'], j['layout'], g_min, t).items()
                          if k in ('reliability', 'power', 'withheld')}
            rows.append(row)
        final = _params(learner, calibration, g_min, targets, taus_db, slack)
        final_est = [predict_on(learner, t['judged']) for t in tasks]
        test_est = predict_on(learner, test)
        js = [t['judged'] for t in tasks]
        g, lay, ep = ([d[k] for d in js] for k in ('gain', 'layout', 'episode'))
        result = {}
        for t in targets:
            result[t] = dict(
                adapted=_judged([e for e, _ in est], g, lay, ep, [p[t] for _, p in est], g_min, t, members),
                fixed=_judged([e for e, _ in est], g, lay, ep, [start[t]] * len(est), g_min, t, members),
                final=_judged(final_est, g, lay, ep, [final[t]] * len(js), g_min, t, members),
                test=judge(*transmissions(test_est, g_min, final[t]['margin_db'], final[t]['tau_db']), test['gain'],
                           test['layout'], g_min, t, episode=test['episode']),
                margin_db=final[t]['margin_db'], tau_db=final[t]['tau_db'])
        out[name] = dict(tasks=rows, targets=result,
                         prediction={g: prediction_errors(np.concatenate([e for (e, _), t in zip(est, tasks) if t['layout'] in ids]),
                                                          np.concatenate([t['judged']['gain'] for t in tasks if t['layout'] in ids]))
                                     for g, ids in members.items() if any(t['layout'] in ids for t in tasks)})
    return out


def evaluate_continual(
    data: dict, # {'train', 'calibration', 'test'}: `build_dataset`'s
    stream: dict, # The stream's layouts' samples (e.g. the deployment layouts'), with `episode`
    groups: dict, # Group -> its layouts, in the order they come (`stream_order`)
    g_min: float, # The weakest servable gain
    model: dict | None = None, # The learning predictor: `fit_model`'s options (kind lstm or gru, quantile, steps, sequence)
    methods=METHODS, # Of `ContinualLearner`
    trees: dict | None = None, # The trees too (frozen, and fit again every `trees_every` layouts): `fit_model`'s options; None: not
    trees_every: int | None = 10,
    order: str = 'groups', # `stream_order`'s
    runs: int = 1, # Stream orders (seed + run): the predictor is pre-trained once
    update_episodes: int = 2, # Of each layout, learned from; the others judged
    targets=(0.9, 0.95, 0.99), taus_db=TAUS_DB, slack: float = 0.0,
    recalibrate_every: int = 1,
    seed: int = 0,
    **learning, # `ContinualLearner`'s options (capacity, epochs, batch_size, lr, replay_weight, ewc, si, si_xi, lwf, clip, device)
) -> dict:
    """`run_stream` of every method over `runs` orders of the stream: `runs` (each one's order and results), and the
    names of the learning predictor and of the trees."""
    spec = {**dict(kind='lstm', quantile=0.05, steps=False), **(model or {})}
    if spec['kind'] == 'trees':
        raise ValueError('the learning predictor is a sequence predictor (lstm or gru); the trees: trees=...')
    base = fit_model(data['train'], **spec)
    name = predictor_name(spec['kind'], spec['quantile'], spec['steps'])
    tree_spec = None if trees is None else dict(kind='trees', quantile=spec['quantile'], **trees)
    tree = fit_model(data['train'], **tree_spec) if tree_spec else None
    out = dict(predictor=name, trees=predictor_name('trees', spec['quantile']) if tree else None, runs=[])
    for run in range(runs):
        order_r = stream_order(groups, order, seed + run)
        tasks = layout_stream(stream, order_r, update_episodes)
        learners = {m: ContinualLearner(base, m, data['train'], seed=seed + run, **learning) for m in methods}
        if tree is not None:
            learners['trees, frozen'] = TreesRefit(tree, data['train'], spec['quantile'], None)
            if trees_every:
                learners[f'trees, refit every {trees_every}'] = TreesRefit(copy.deepcopy(tree), data['train'], spec['quantile'],
                                                                         trees_every, **(trees.get('trees') or {}))
        out['runs'].append(dict(order=order_r, results=run_stream(learners, tasks, data['calibration'], data['test'], g_min,
                                                                  targets, taus_db, slack, groups, recalibrate_every)))
    return out


def continual_summary(results: dict, target: float, kinds=('adapted', 'fixed', 'final', 'test')) -> str:
    "A text table: per method, at `target`, the reliability and power of each kind (pooled over the stream), mean over runs."
    runs = results['runs']
    methods = list(runs[0]['results'])
    w = max(map(len, methods)) + 2
    lines = [f"{results['predictor']} learning over the stream (trees: {results['trees']}), target {100 * target:g}%, "
             f"mean over {len(runs)} orders: reached / power (× perfect CSI)",
             f"{'':{w}s}" + ''.join(f'{k:>22s}' for k in kinds)]
    for m in methods:
        row = f'{m:{w}s}'
        for k in kinds:
            get = lambda r: r['results'][m]['targets'][target][k]
            o = [get(r) if k == 'test' else get(r)['all'] for r in runs]
            row += f"{100 * np.mean([x['reliability'] for x in o]):13.1f}% {np.mean([x['power'] for x in o]):6.2f}×"
        lines.append(row)
    lines.append('adapted: each layout sent after learning from its feedback, κ and τ chosen again; fixed: the start\'s κ and τ; '
                 'final: the last predictor on every layout of the stream; test: on the test layouts')
    return '\n'.join(lines)

In [ ]:
show_doc(run_stream)

In [ ]:
show_doc(evaluate_continual)

In [ ]:
#| export
KINDS = ('adapted', 'fixed', 'final', 'test')

def continual_rows(results: dict) -> tuple[list, list]:
    """Rows for tables: per layout (run, method, task, layout, group, transmissions, the errors before and after, and per
    target the reliability, power, withheld), and per method's metric (run, method, target, kind, group, reliability,
    power, withheld, layout_p10, layout_below)."""
    tasks, methods = [], []
    for r, run in enumerate(results['runs']):
        for m, res in run['results'].items():
            for row in res['tasks']:
                tasks.append([r, m, row['task'], row['layout'], row['group'], row['transmissions'],
                              *(row[w][k] for w in ('before', 'after') for k in ('bias_db', 'spread_db', 'coverage')),
                              *(row[t][k] for t in res['targets'] for k in ('reliability', 'power', 'withheld'))])
            for t, o in res['targets'].items():
                for kind in KINDS:
                    for g, v in ({'test': o[kind]} if kind == 'test' else o[kind]).items():
                        methods.append([r, m, t, kind, g, *(v.get(k) for k in ('reliability', 'power', 'withheld', 'layout_p10', 'layout_below'))])
    return tasks, methods

def task_columns(results: dict) -> list[str]:
    "The columns of `continual_rows`' per-layout rows."
    targets = list(results['runs'][0]['results'][next(iter(results['runs'][0]['results']))]['targets'])
    return ['run', 'method', 'task', 'layout', 'group', 'transmissions',
            *(f'{w}_{k}' for w in ('before', 'after') for k in ('bias_db', 'spread_db', 'coverage')),
            *(f'{100 * t:g}%_{k}' for t in targets for k in ('reliability', 'power', 'withheld'))]

METHOD_COLUMNS = ['run', 'method', 'target', 'kind', 'group', 'reliability', 'power', 'withheld', 'layout_p10', 'layout_below']

def plot_continual(results: dict, target: float, path=None, window: int = 10, ax=None):
    """Per method, the power each layout of the stream takes at `target` (× perfect CSI, after learning from it, mean over
    the runs, a running mean over `window` layouts, centred); the groups' boundaries (the first run's order) dotted."""
    import matplotlib.pyplot as plt
    runs = results['runs']
    fig = None
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 4.5))
    for m in runs[0]['results']:
        p = np.nanmean([[row[target]['power'] for row in r['results'][m]['tasks']] for r in runs], 0)
        w = max(1, min(window, len(p) // 3))                      # a short stream: a shorter window
        y = np.convolve(p, np.ones(w) / w, mode='valid')
        ax.plot(np.arange(len(y)) + (w - 1) / 2, y, label=m, marker='.' if len(y) < 20 else None,
                lw=1.5 if not m.startswith('trees') else 1.0, ls='-' if not m.startswith('trees') else '--')
    groups = [row['group'] for row in runs[0]['results'][next(iter(runs[0]['results']))]['tasks']]
    for i in range(1, len(groups)):
        if groups[i] != groups[i - 1]:
            ax.axvline(i - 0.5, color='grey', ls=':', lw=1)
    ax.set(xlabel='layout of the stream', ylabel=f'power for {100 * target:g}% (× perfect CSI)',
           title=f"{results['predictor']} learning over the stream")
    ax.legend(fontsize=7, ncol=2)
    if fig is not None and path is not None:
        fig.tight_layout()
        fig.savefig(path, dpi=120)
        plt.close(fig)
    return ax

## Example

The stand-in channel of `comm.evaluation` (free-space loss and 6 dB per wall crossed, as its large-scale channel, Rayleigh fading on top): an LSTM
pre-trained on 20 layouts, then a stream of 8 more in two groups, each layout's first episode learned from.

In [ ]:
import tempfile
from pathlib import Path

from c3jepa_wm.comm.channel_data import PairCache, build_dataset, collect_trajectories, trace_visited_pairs, wall_geometry
from c3jepa_wm.comm.power_control import Budget

In [ ]:
env_cfg = {'_target_': 'stable_marl.envs.multigrid.FindGoalEnv', 'agents': 2, 'size': 9, 'num_obstacles': 0, 'max_steps': 60}
tmp = Path(tempfile.mkdtemp())

def stand_in(walls):
    "A tracer for `PairCache`: free-space loss at 2.4 GHz (2 m cells) and 6 dB per wall crossed."
    def tracer(tx, rxs):
        return [0.125 / (4 * np.pi * 2 * d) * 10 ** (-6 * w / 20) for d, w in (wall_geometry(walls, tx, rx) for rx in rxs)]
    return tracer

collect_trajectories(env_cfg, range(36), tmp, episodes=4, max_steps=60)
for seed in range(36):
    trace_visited_pairs(env_cfg, tmp, [seed], tmp, tracer=stand_in(PairCache(tmp / 'unused.npz', env_cfg, seed).walls), progress=False)
data = {k: build_dataset(tmp, tmp, s, channel='large_scale', fading='rician') for k, s in dict(train=range(20), calibration=range(20, 24), test=range(24, 28)).items()}
stream = build_dataset(tmp, tmp, range(28, 36), channel='large_scale', fading='rician')
groups = {'first': list(range(28, 32)), 'then': list(range(32, 36))}
g_min = Budget(snr_target_db=10, noise_dbm=-94, max_power_dbm=10).g_min
model = fit_model(data['train'], 'lstm', 0.05, sequence=dict(hidden=32, head=32, epochs=10))

In [ ]:
results = evaluate_continual(data, stream, groups, g_min, model=dict(kind='lstm', sequence=dict(hidden=32, head=32, epochs=10)),
                             trees=dict(trees=dict(max_iter=100)), trees_every=4, runs=2, update_episodes=1, targets=(0.95,),
                             epochs=5, capacity=500)
print(continual_summary(results, 0.95))
plot_continual(results, 0.95, window=3);

In [ ]:
# the reservoir: every sample seen equally likely kept; loss-aware, the hard ones kept
buf, lars = ReplayBuffer(100, seed=0), ReplayBuffer(100, lars=True, seed=0)
X, gain, loss = np.arange(2000, dtype=float)[:, None], np.ones(2000), np.r_[np.full(1000, 10.0), np.full(1000, 0.01)]
for b in (buf, lars):
    b.add(X, gain, loss)
assert len(buf) == len(lars) == 100 and buf.seen == 2000
assert 30 < (buf.arrays['X'][:, 0] < 1000).sum() < 70                  # about half of each half
assert (lars.arrays['X'][:, 0] < 1000).sum() > 70                       # the high-loss half stays
full = ReplayBuffer(None)
full.add(X, gain, loss)
assert len(full) == 2000 and full.sample(5)['X'].shape == (5, 1)

In [ ]:
# the learners: frozen never changes; the others learn; each method's own state
d = data['train']
x = d['X'][:200]
pretrained = model.predict(x)
for method in METHODS:
    learner = ContinualLearner(model, method, d, epochs=3, capacity=300)
    before = learner.predict(x)
    learner.learn(stream)
    changed = not np.allclose(learner.predict(x), before)
    assert changed == (method != 'frozen'), method
    assert np.allclose(model.predict(x), pretrained)                    # the pre-trained one untouched
    if method == 'ewc':
        assert len(learner.anchors) == 2
    if method in ('replay', 'lars'):
        assert len(learner.buffer) == 300
    if method == 'joint':
        assert len(learner.buffer) == len(d['gain']) + len(stream['gain'])
    if method == 'si':
        assert all((v >= 0).all() for v in learner.omega.values())
for bad in (lambda: ContinualLearner(model, 'rnn'), lambda: ContinualLearner(model, 'replay'),
            lambda: ContinualLearner(fit_model(d, 'trees', trees=dict(max_iter=20)), 'frozen')):
    try:
        bad()
        raise AssertionError('no error')
    except ValueError:
        pass

In [ ]:
# the stream: frozen sends each layout as it would without learning (adapted = fixed = final); every layout judged once
r = results['runs'][0]['results']
assert set(r) == {*METHODS, 'trees, frozen', 'trees, refit every 4'}
assert sorted(stream_order(groups, 'random', 0)) == sorted(stream_order(groups, 'groups', 0)) == list(range(28, 36))
assert set(stream_order(groups, 'groups', 0)[:4]) == set(groups['first'])   # group after group
f = r['frozen']['targets'][0.95]
for g in ('all', 'first', 'then'):
    assert f['adapted'][g] == f['fixed'][g] == f['final'][g]
n = sum(row['transmissions'] for row in r['frozen']['tasks'])
assert n == (stream['episode'] >= 1).sum() == f['adapted']['all']['transmissions']
assert [row['layout'] for row in r['frozen']['tasks']] == results['runs'][0]['order']
tasks, methods = continual_rows(results)
assert len(tasks[0]) == len(task_columns(results)) and len(methods[0]) == len(METHOD_COLUMNS)